In [1]:
import scanpy as sc
import anndata as ad
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.colors import ListedColormap, rgb2hex
import numpy as np
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
import numpy as np
from sklearn.metrics import jaccard_score
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
adata = sc.read_h5ad('/data/work/05.cluster/FuseMap/0106/Hippocampus_latent_embeddings_all_single_pretrain/dmt_leiden_20250108_1.h5ad')
adata

AnnData object with n_obs × n_vars = 1112773 × 33326
    obs: 'dnbCount', 'area', 'orig.ident', 'x', 'y', 'region', 'n_counts', 'region_h2', 'Tangram_1119_celltype', 'Tangram_1119_celltype_main_frac', 'region_hip', 'slice_code', 'sub_region', 'dmt_leiden', 'dmt_leiden_merge'
    uns: 'dmt_leiden_colors', 'dmt_nn', 'leiden', 'slice_code_colors'
    obsm: 'X_dmt', 'X_dmt_highdim', 'align_spatial_2d', 'align_spatial_3d', 'cell_border', 'latent_embeddings_all_single_pretrain', 'latent_embeddings_all_spatial_pretrain', 'spatial', 'spatial_division'
    obsp: 'dmt_nn_connectivities', 'dmt_nn_distances'

In [3]:
names = [
    '12_B03605F3G5_WT202403310048.h5ad',
    '13_B03612A1C3_WT202403310056.h5ad',
    '14_A03591A1C3_WT202403310045.h5ad',
    '16_A03592A4C6_WT202403310044.h5ad',
    '18_B03602C4D6_WT202405020031.h5ad',
    '20_B03606F3G5_WT202405020032.h5ad',
    '22_B03606C4E6_WT202403310050.h5ad',
    '23_B03609A4D6_WT202404150263.h5ad',
    '27_B03610C1E3_WT202403310051.h5ad',
    '31_B03619A1D3_WT202403310052.h5ad',
    '35_B03619E4G6_WT202403310053.h5ad',
    '39_A03589A1D4_WT202403310046.h5ad',
    '43_A03590E1G4_WT202403310064.h5ad',
    '47_A03593C1F3_WT202403310068.h5ad',
    'B03607C4E6_WT2024071214941.h5ad',
]

In [4]:
dic = {
    '0': 'hip_0',
    '1': 'hip_1',
    '2': 'hip_2',
    '3': 'hip_1',
    '4': 'hip_3',
    '5': 'hip_4',
    '6': 'hip_1',
    '7': 'hip_0',
    '8': 'hip_5',
    '9': 'hip_6',
    '10': 'hip_5',
    '11': 'hip_6',
    '12': 'hip_7',
    '13': 'hip_2',
    '14': 'hip_1',
    '15': 'hip_8',
    '16': 'hip_6',
    '17': 'hip_8',
    '18': 'hip_6',
    '19': 'hip_2',
    '20': 'hip_2',
    '21': 'hip_3',
    '22': 'hip_5',
    '23': 'hip_4',
    '24': 'NA',
    '25': 'hip_8',
    '26': 'hip_6',
    '27': 'hip_4',
    '28': 'hip_4',
    '29': 'hip_8',
    '30': 'hip_4',
    '31': 'NA',
}
adata.obs['dmt_leiden_merge'] = [dic[i] for i in adata.obs['dmt_leiden']]


In [5]:
sc.pp.filter_genes(adata, min_cells = 100)
adata

AnnData object with n_obs × n_vars = 1112773 × 24269
    obs: 'dnbCount', 'area', 'orig.ident', 'x', 'y', 'region', 'n_counts', 'region_h2', 'Tangram_1119_celltype', 'Tangram_1119_celltype_main_frac', 'region_hip', 'slice_code', 'sub_region', 'dmt_leiden', 'dmt_leiden_merge'
    var: 'n_cells'
    uns: 'dmt_leiden_colors', 'dmt_nn', 'leiden', 'slice_code_colors'
    obsm: 'X_dmt', 'X_dmt_highdim', 'align_spatial_2d', 'align_spatial_3d', 'cell_border', 'latent_embeddings_all_single_pretrain', 'latent_embeddings_all_spatial_pretrain', 'spatial', 'spatial_division'
    obsp: 'dmt_nn_connectivities', 'dmt_nn_distances'

In [6]:
adatas = []
for i in set(adata.obs['slice_code']):
    temp = adata[adata.obs['slice_code'] == i].copy()
    sc.pp.normalize_total(temp)
    sc.pp.log1p(temp)
    sc.pp.scale(temp, zero_center=False, max_value=10)
    adatas.append(temp)
adata = ad.concat(adatas)

In [7]:
adata.obs_names_make_unique()

In [8]:
adata_2 = adata[adata.obs.sample(frac = 0.3).index].copy()

In [9]:
sc.tl.rank_genes_groups(adata_2, groupby = 'dmt_leiden', pts = True, method = 'wilcoxon', groups = ['9', '11', '16', '18', '26'])

In [92]:
sc.get.rank_genes_groups_df(adata_2, '6').columns

Index(['names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj',
       'pct_nz_group', 'pct_nz_reference'],
      dtype='object')

In [12]:
sc.get.rank_genes_groups_df(adata_2, '16').sort_values('logfoldchanges', ascending = False)[: 20]#.names.tolist()

,names,scores,logfoldchanges,pvals,pvals_adj,pct_nz_group,pct_nz_reference
458,OVOL1,0.078293,2.236594,0.937595,0.999988,0.000567,0.000121
308,OXT,0.103307,2.191587,0.917719,0.999988,0.000756,0.000167
913,AL590428.1,0.050026,2.028979,0.960101,0.999988,0.000378,0.000093
205,AC097505.1,0.137301,2.023860,0.890793,0.999988,0.001040,0.000257
327,AL355773.1,0.098968,1.983691,0.921164,0.999988,0.000756,0.000192
928,AC024559.2,0.049484,1.981651,0.960534,0.999988,0.000378,0.000096
929,BARX1,0.049484,1.981651,0.960534,0.999988,0.000378,0.000096
930,AC004895.1,0.049484,1.981651,0.960534,0.999988,0.000378,0.000096
678,AF131216.1,0.061719,1.972877,0.950786,0.999988,0.000473,0.000121
949,AL355472.4,0.048942,1.935825,0.960966,0.999988,0.000378,0.000099
